# STEP 3 — 인과관계 추출 → 표준화 → 집계 (재현용) ⭐

파이프라인의 **핵심**. 관련 논문 초록에서 GPT가 **(원인 → 관계 → 결과)** 삼중항을 뽑고, 표현을 통일(표준화)한 뒤, 같은 관계가 몇 번 나왔는지 세어 **엣지 표** `step3_edges.csv` 를 만듭니다.

| | |
|---|---|
| **입력** | `step2_filtered.csv` (relevant=1) |
| **출력** | `step3_edges.csv` (cause, category_cause, effect, category_effect, relationship, frequency, num_papers) |
| **필요** | OpenAI API 키 |

```
Stage 2a 추출   초록 → (원인, 관계, 결과) 삼중항
Stage 2b 표준화  같은 뜻 표현을 표준 용어로 통일
Stage 2c 집계   (원인,관계,결과)별 빈도·논문수 → 엣지 표
```
> 💰 기본 모델은 **gpt-5-mini**(원본과 동일). 비용을 위해 기본 `LIMIT=30` 편만 처리합니다(키우려면 값↑). 더 싸게 하려면 `MODEL='gpt-4o-mini'`.

**쉽게 말하면:** GPT가 논문을 정독하며 *"무엇이 무엇에 영향을 준다"* 는 문장을 찾아 **(원인 → 관계 → 결과)** 카드로 만들고, 같은 카드가 몇 번 나왔는지 세는 단계예요.

In [ ]:
!pip install -q openai pandas
print('준비 완료')

In [ ]:
import os, re, json, time, pandas as pd
from collections import Counter, defaultdict
from getpass import getpass

OPENAI_KEY = ''
try:
    from google.colab import userdata
    OPENAI_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_KEY:
    OPENAI_KEY = getpass('OpenAI API key: ').strip()
from openai import OpenAI
client = OpenAI(api_key=OPENAI_KEY)

MODEL           = 'gpt-5-mini'    # 원본 연구와 동일 (비용 줄이려면 'gpt-4o-mini')
LIMIT           = 30              # 처리할 관련 논문 수 (비용)
NORM_BATCH_SIZE = 80              # 표준화 배치 크기
print('준비 완료, 모델:', MODEL)

In [ ]:
# 입력: STEP 2의 관련 논문
if not os.path.exists('step2_filtered.csv'):
    raise FileNotFoundError('step2_filtered.csv 가 없습니다. STEP 2를 먼저 실행하세요.')
df_f = pd.read_csv('step2_filtered.csv')
df_rel = df_f[df_f['relevant'] == 1].head(LIMIT).reset_index(drop=True)
print(f'관련 논문 {int(df_f["relevant"].sum())}편 중 앞 {len(df_rel)}편 처리')

In [ ]:
CANONICAL_NODES = """
aggregation, viscosity, particle formation, colloidal stability, self-association, solubility,
hmw species, monomer content, adsorption, turbidity, opalescence,
oxidation, deamidation, isomerization, fragmentation, degradation, charge variants, glycosylation,
disulfide bond, conformational stability, thermal stability, melting temperature, protein unfolding,
storage stability, in vivo stability, binding activity, binding affinity, potency, immunogenicity,
pharmacokinetics, half-life, clearance, efficacy, higher-order structure, amino acid sequence,
concentration, ph, low ph, polysorbate 80, polysorbate 20, sucrose, trehalose, arginine, histidine,
surfactant, excipients, buffer composition, ionic strength, thermal stress, agitation, freeze-thaw,
light exposure, oxidative stress, lyophilization, shear stress, air-water interface
"""
canonical_list = [t.strip() for t in CANONICAL_NODES.split(',') if t.strip()]
print(f'표준 노드 {len(canonical_list)}개')

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an expert pharmaceutical scientist specializing in monoclonal antibody (mAb) formulation and stability.

Extract MECHANISTIC relationships relevant to mAb STABILITY from the abstract.
These are directional associations explicitly described by the authors (physical/chemical/biological mechanisms), not statistically inferred effects.

=== 6 NODE CATEGORIES ===
sequence       : amino acid sequence features (CDR residue, mutation, glycosylation site)
structure      : 3D structural features (hydrophobic patch, disulfide bond, Fc domain)
formulation    : formulation components (PS80, sucrose, pH, arginine, concentration)
stress         : stress conditions (thermal stress, freeze-thaw, agitation, light)
stability      : stability properties & degradation (aggregation, deamidation, Tm, viscosity)
quality_outcome: measurable quality results (particle formation, immunogenicity, charge variant, PK)

=== RELATION TYPES ===
Positive: stabilizes, inhibits, prevents, decreases, shields
Negative: destabilizes, promotes, increases, induces, aggregates, oxidizes, deamidates, isomerizes, fragments, unfolds, adsorbs, precipitates, degrades
Neutral : correlates, modifies, binds, requires

Use specific terms: "low pH" not "pH", "elevated temperature" not "temperature".

Return ONLY JSON: {"relations":[{"cause":"...","category_cause":"...","effect":"...","category_effect":"...","relationship":"...","confidence":"high/medium/low","evidence":"key phrase (max 30 words)"}]}
No markdown, no text outside the JSON."""

def clean_text(t):
    if not isinstance(t, str): t = str(t)
    rep = {'\u00b0':' deg ','\u00b1':'+/-','\u2265':'>=','\u2264':'<=','\u00b5':'u','\u03b1':'alpha','\u03b2':'beta','\u2013':'-','\u2014':'-'}
    for k,v in rep.items(): t = t.replace(k, v)
    return t.encode('ascii','ignore').decode('ascii')

def extract_relations(title, abstract):
    user = f'Title: {clean_text(title)}\nAbstract: {clean_text(abstract)}'
    for attempt in range(3):
        try:
            r = client.chat.completions.create(model=MODEL,
                messages=[{'role':'system','content':EXTRACTION_SYSTEM_PROMPT},{'role':'user','content':user}],
                max_completion_tokens=16000)   # gpt-5-mini 추론+JSON 여유 (원본 20000)
            txt = r.choices[0].message.content.replace('```json','').replace('```','').strip()
            return json.loads(txt).get('relations', [])
        except Exception as e:
            if attempt < 2: time.sleep(2)
            else: print('  추출 실패:', str(e)[:60]); return []
print('Stage 2a 함수 정의 완료')

In [ ]:
# Stage 2a 실행: 초록 → 관계 추출
raw = []
for idx, row in df_rel.iterrows():
    rels = extract_relations(row['title'], row['abstract'])
    for rel in rels:
        rel['pmid'] = str(row['pmid'])
        raw.append(rel)
    if (idx+1) % 10 == 0: print(f'  {idx+1}/{len(df_rel)} 편, 누적 관계 {len(raw)}개')
    time.sleep(0.2)
df_raw = pd.DataFrame(raw)
print(f'\n✅ Stage 2a: 원시 관계 {len(df_raw)}개')
df_raw.head(6)

In [ ]:
NORMALIZATION_SYSTEM_PROMPT = """You are an expert in mAb stability terminology.
Map each input term to the most appropriate canonical term from the reference list.
If none fits, keep the original but make it concise and lowercase.

CANONICAL REFERENCE TERMS:
""" + ', '.join(canonical_list) + """

RULES:
1. Map synonyms/variants to the canonical term (e.g. "protein aggregation" -> "aggregation").
2. Keep condition-specific terms distinct (e.g. "low pH" and "high pH" stay separate).
3. Use lowercase, concise English.

Return ONLY JSON: {"original term": "canonical term", ...}  No markdown."""

def normalize_terms(terms):
    uniq = sorted(set(str(t) for t in terms if t and str(t) != 'nan'))
    mapping = {}
    for i in range(0, len(uniq), NORM_BATCH_SIZE):
        batch = uniq[i:i+NORM_BATCH_SIZE]
        prompt = 'Normalize these mAb stability terms:\n' + '\n'.join(f'- {t}' for t in batch)
        for attempt in range(3):
            try:
                r = client.chat.completions.create(model=MODEL,
                    messages=[{'role':'system','content':NORMALIZATION_SYSTEM_PROMPT},{'role':'user','content':prompt}],
                    max_completion_tokens=12000)
                txt = r.choices[0].message.content.replace('```json','').replace('```','').strip()
                mapping.update(json.loads(txt)); break
            except Exception as e:
                if attempt < 2: time.sleep(2)
                else:
                    for t in batch: mapping[t] = t
        time.sleep(0.5)
    return mapping

# Stage 2b 실행
if len(df_raw):
    cmap = normalize_terms(df_raw['cause'].tolist())
    emap = normalize_terms(df_raw['effect'].tolist())
    df_raw['cause']  = df_raw['cause'].map(lambda x: cmap.get(str(x), str(x)).lower().strip())
    df_raw['effect'] = df_raw['effect'].map(lambda x: emap.get(str(x), str(x)).lower().strip())
    print(f'✅ Stage 2b 표준화 완료 (cause {df_raw["cause"].nunique()}종, effect {df_raw["effect"].nunique()}종)')
else:
    print('추출된 관계가 없습니다. LIMIT를 늘리거나 STEP 1 검색어를 넓혀보세요.')

In [ ]:
# Stage 2c 집계: (cause, effect, relationship)별 빈도·논문수
agg = defaultdict(lambda: {'count':0,'pmids':set(),'cc':Counter(),'ce':Counter(),'conf':Counter(),'ev':[]})
for _, r in df_raw.iterrows():
    key = (str(r['cause']), str(r['effect']), str(r.get('relationship','correlates')))
    a = agg[key]; a['count'] += 1; a['pmids'].add(str(r.get('pmid','')))
    a['cc'][str(r.get('category_cause',''))] += 1; a['ce'][str(r.get('category_effect',''))] += 1
    a['conf'][str(r.get('confidence','medium'))] += 1
    ev = str(r.get('evidence',''))[:120]
    if ev and ev != 'nan': a['ev'].append(ev)
rows = []
for (cause, effect, rel), a in agg.items():
    rows.append({'cause':cause,'category_cause':a['cc'].most_common(1)[0][0] if a['cc'] else '',
                 'effect':effect,'category_effect':a['ce'].most_common(1)[0][0] if a['ce'] else '',
                 'relationship':rel,'frequency':a['count'],'num_papers':len(a['pmids']),
                 'main_confidence':a['conf'].most_common(1)[0][0] if a['conf'] else 'medium',
                 'sample_evidence':a['ev'][0] if a['ev'] else ''})
edges = pd.DataFrame(rows).sort_values('frequency', ascending=False).reset_index(drop=True)
edges.to_csv('step3_edges.csv', index=False, encoding='utf-8-sig')
print(f'✅ 저장: step3_edges.csv | 고유 엣지 {len(edges)}개')
edges.head(12)

### ✅ STEP 3 완료
`step3_edges.csv` 가 **그래프의 재료이자 GATED 모델의 학습 데이터**입니다.

- **그래프만 그리려면:** `02_draw_graph.ipynb` 에 이 CSV를 넣으세요.
- **모델까지:** `step4_gated.ipynb` 로.

> ⚠️ 소규모(30편)라 엣지가 적습니다. 원본 논문 수준(32,939 엣지)을 보려면 LIMIT·검색어를 크게 키우거나, STEP 4에서 제공되는 **공개 엣지 데이터**로 학습하세요.